# Notebook 03: Sistema de Priorização

## Objetivo
Criar uma métrica (Score de Prioridade) que combine o modelo preditivo, regras de negócio territoriais e impacto climático. Validar este sistema através de uma Lift Curve.

### Regra de Negócio Proposta:
`Score = (Probabilidade de Atraso * W1) + (Fator Territorial * W2) + (Fator Climático * W3)`

- **Probabilidade de Atraso**: 1 - P(Resolvido em 7 dias) do nosso modelo LightGBM.
- **Fator Territorial**: Penalidade maior (ex: +0.2) para áreas vulneráveis como AP 3 e AP 5.
- **Fator Climático**: Se for chamado de Alagamento ou Poda em dia de chuva forte, +0.3.

In [ ]:
import sys
import os
sys.path.append(os.path.abspath('../src'))

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from data_fetcher import DataFetcher
from features import create_stratified_sample, temporal_train_test_split
from model_utils import plot_lift_curve

# Carrega as previsões salvas pelo Notebook 02 no conjunto de teste real
try:
    df_sim = pd.read_csv('../data/test_predictions.csv')
    print(f"Dados reais carregados com {len(df_sim)} registros.")
except FileNotFoundError:
    print("Arquivo '../data/test_predictions.csv' não encontrado. Gerando dados de demonstração.")
    np.random.seed(42)
    n = 10000
    df_sim = pd.DataFrame({
        'id_chamado': range(n),
        'area_planejamento': np.random.choice(['AP 1', 'AP 2', 'AP 3', 'AP 4', 'AP 5'], n),
        'tipo': np.random.choice(['Buraco', 'Alagamento', 'Poda de Árvore', 'Iluminação'], n),
        'chuva_dia': np.random.choice([0, 1], n, p=[0.8, 0.2]),
        'prob_resolvido_7d': np.random.uniform(0.1, 0.9, n),
        'realmente_atrasou': np.random.choice([0, 1], n, p=[0.7, 0.3])
    })
    df_sim['prob_atraso'] = 1 - df_sim['prob_resolvido_7d']

## 1. Cálculo do Score de Prioridade

In [ ]:
def calculate_priority_score(row):
    score = row['prob_atraso'] * 0.5  # Peso do modelo preditivo: 50%
    
    # Fator Territorial (30%)
    if row['area_planejamento'] in ['AP 3', 'AP 5']:
        score += 0.3
    elif row['area_planejamento'] in ['AP 1']:
        score += 0.1
        
    # Fator Climático (20%)
    if row['chuva_dia'] == 1 and row['tipo'] in ['Alagamento', 'Poda de Árvore']:
        score += 0.2
        
    # Normalização entre 0 e 1 (Aproximada)
    return min(max(score, 0.0), 1.0)

df_sim['score_prioridade'] = df_sim.apply(calculate_priority_score, axis=1)
df_sim = df_sim.sort_values(by='score_prioridade', ascending=False).reset_index(drop=True)
df_sim.head()

## 2. Validação de Negócio (Gain Chart / Lift Curve)
Vamos comparar: se a prefeitura atender apenas os 20% primeiros chamados do nosso ranking, qual a proporção de atrasos reais que nós interceptamos comparado à seleção aleatória?

In [ ]:
plot_lift_curve(df_sim['realmente_atrasou'], df_sim['score_prioridade'])

### Conclusão Sênior:
Com este sistema Híbrido (Machine Learning + Regras de Especialista), a gestão pública consegue focar os recursos limitados onde eles são estatisticamente mais propensos a falhar, respeitando ao mesmo tempo diretrizes de equidade territorial (peso maior para APs vulneráveis) e mitigação de risco imediato (chuvas).